# Aula 08 — Reducers: a regra de mesclagem do estado

**Objetivo:** entender quando o `overwrite` quebra, o que é um reducer e por
que `add_messages` tem 3 poderes. Ao final você deve ler um `InvalidUpdateError`
e saber exatamente qual `Annotated` resolve.

**Pré-requisitos:** Aula 02 (overwrite: último ganha) e Aula 03 (`add_messages`).

**Como usar:** rode em ordem. Autocontido, 100% offline (sem LLM, sem chave).
Comece pela CÉLULA 0.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph
print('dependências prontas.')

## 1. Analogia: dois caixas na mesma comanda

Na lanchonete, a cozinha e o bar terminam juntos e anotam na mesma comanda
**ao mesmo tempo**. Sem regra de mesclagem, um apaga o outro. Com regra
('anexe os dois itens'), a comanda fica completa.

| Lanchonete | No LangGraph | O que é de verdade |
|---|---|---|
| uma anotação por vez | **overwrite** (padrão) | cada chave aceita 1 valor por etapa |
| duas anotações juntas | **fan-out paralelo** | dois nós escrevem na mesma etapa |
| regra da comanda | **reducer** | função `(atual, novo) → mesclado`, via `Annotated` |
| 'anexe, não apague' | **`operator.add`** | o reducer mais comum (listas e números) |

Regra de bolso: **sequência** tolera overwrite (último ganha); **paralelo**
exige reducer — ou o grafo aborta.

In [ ]:
# CÉLULA 1 — sequência ok, paralelo quebra (rode e compare)
from typing import TypedDict
from langgraph.graph import START, END, StateGraph
from langgraph.errors import InvalidUpdateError

class Estado(TypedDict):
    foo: int

# sequência: overwrite funciona, último ganha
b = StateGraph(Estado)
b.add_node('n1', lambda s: {'foo': s['foo'] + 1})
b.add_edge(START, 'n1'); b.add_edge('n1', END)
print('sequência:', b.compile().invoke({'foo': 1}))

# fan-out: n2 e n3 escrevem NA MESMA etapa
bf = StateGraph(Estado)
bf.add_node('n1', lambda s: {'foo': s['foo'] + 1})
bf.add_node('n2', lambda s: {'foo': s['foo'] + 1})
bf.add_node('n3', lambda s: {'foo': s['foo'] + 1})
bf.add_edge(START, 'n1'); bf.add_edge('n1', 'n2'); bf.add_edge('n1', 'n3')
bf.add_edge('n2', END); bf.add_edge('n3', END)
try:
    bf.compile().invoke({'foo': 1})
    print('fan-out rodou?! (não deveria)')
except InvalidUpdateError as erro:
    print('fan-out -> InvalidUpdateError:')
    print(' ', str(erro).splitlines()[0])

## 2. Conceito: reducer é regra de mesclagem por etapa

- O grafo executa em **etapas** (*steps*): todos os nós prontos na etapa rodam
  'juntos'. Em sequência, cada etapa tem 1 escritor (overwrite basta). Em
  fan-out, uma etapa tem N escritores — e aí a chave precisa de regra.
- **`Annotated[list, operator.add]`** diz: 'nesta chave, some os updates em vez
  de escolher um'. O `Annotated` é o gancho onde o LangGraph pendura a regra.
- O reducer roda **entre etapas**: junta o valor atual com cada update, na ordem.
- `InvalidUpdateError: Can receive only one value per step` é o erro mais
  informativo do curso: ele diz a chave (`At key 'foo'`) e a solução
  (`Use an Annotated key`).

In [ ]:
# CÉLULA 2 — fan-out com operator.add (rode e decifre o [1, 2, 3, 3])
from operator import add
from typing import Annotated, TypedDict
from langgraph.graph import START, END, StateGraph

class EstadoLista(TypedDict):
    foo: Annotated[list, add]

b = StateGraph(EstadoLista)
b.add_node('n1', lambda s: {'foo': [s['foo'][-1] + 1]})
b.add_node('n2', lambda s: {'foo': [s['foo'][-1] + 1]})
b.add_node('n3', lambda s: {'foo': [s['foo'][-1] + 1]})
b.add_edge(START, 'n1'); b.add_edge('n1', 'n2'); b.add_edge('n1', 'n3')
b.add_edge('n2', END); b.add_edge('n3', END)
out = b.compile().invoke({'foo': [1]})
print(out)
assert out == {'foo': [1, 2, 3, 3]}, out
print('Leitura: [1] -> n1 anexa 2 -> [1, 2] -> n2 e n3 LEEM [1, 2]')
print('(nós paralelos enxergam o MESMO estado de entrada!) e anexam 3 cada um.')

## 3. Dissecando o `[1, 2, 3, 3]` (o detalhe que mais confunde)

1. Etapa 1: só `n1` roda. Lê `foo=[1]`, devolve `[2]`. Reducer: `[1] + [2]` =
   `[1, 2]`. Estado agora `[1, 2]`.
2. Etapa 2: `n2` e `n3` rodam **juntos**, ambos lendo `foo=[1, 2]`. Cada um
   devolve `[3]` (2+1). Reducer junta em ordem: `[1, 2] + [3] + [3]`.
3. Moral: nós paralelos **não se enxergam** — veem o mesmo retrato do estado
   no início da etapa. Se `n3` precisasse do resultado de `n2`, o desenho
   estaria errado: paralelo é para trabalho **independente**.

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. `None` + `operator.add` = `TypeError`; reducer custom salva

`add` concatena sem perguntar: `lista + None` explode. Reducer custom é uma
função comum `(esquerda, direita) -> mesclado` — use para tolerar `None`,
limitar tamanho ou mesclar ditos 'à sua maneira'.

In [ ]:
# CÉLULA 3 — None quebra o add; custom tolera (100% offline)
from typing import Annotated, TypedDict
from langgraph.graph import START, END, StateGraph

class ComAdd(TypedDict):
    foo: Annotated[list, add]

def reduce_seguro(esquerda, direita):
    if esquerda is None:
        return direita or []
    if direita is None:
        return esquerda
    return esquerda + direita

class ComCustom(TypedDict):
    foo: Annotated[list, reduce_seguro]

b = StateGraph(ComAdd)
b.add_node('n1', lambda s: {'foo': [2]})
b.add_edge(START, 'n1'); b.add_edge('n1', END)
try:
    b.compile().invoke({'foo': None})
except TypeError as erro:
    print('add + None -> TypeError:', erro)

b2 = StateGraph(ComCustom)
b2.add_node('n1', lambda s: {'foo': [2]})
b2.add_edge(START, 'n1'); b2.add_edge('n1', END)
print('custom + None:', b2.compile().invoke({'foo': None}))

### 4b. Os 3 poderes do `add_messages` (revisão com nomes)

1. **Anexar**: mensagens novas entram no fim (o que salva o histórico).
2. **Sobrescrever por `id`**: mensagem com `id` igual a uma existente
   **substitui** (útil para corrigir a última resposta sem apagar o resto).
3. **Remover**: `RemoveMessage(id=...)` apaga pelo `id` (a poda manual do
   histórico — a poda automática com `trim_messages` é a próxima aula).

In [ ]:
# CÉLULA 4 — anexar, sobrescrever por id, remover (100% offline)
from langchain_core.messages import HumanMessage, AIMessage, RemoveMessage
from langgraph.graph.message import add_messages

print('anexar:', [m.content for m in add_messages(
    [HumanMessage(content='oi')], [AIMessage(content='olá')])])

print('sobrescrever id:', [m.content for m in add_messages(
    [AIMessage(content='velha', id='1')],
    [HumanMessage(content='nova mesma id', id='1')])])

print('remover:', [m.content for m in add_messages(
    [AIMessage(content='a', id='1'), HumanMessage(content='b', id='2'),
     AIMessage(content='c', id='3')],
    [RemoveMessage(id='1'), RemoveMessage(id='2')])])

### 4c. `MessagesState` estendido: reducer e overwrite convivem

Cada **canal** tem sua regra: `messages` acumula (reducer herdado),
`resumo` sobrescreve (padrão). Declarar `class Estendido(MessagesState)` com
um campo a mais é o padrão para 'conversa + metadados' (Aula 09 aprofunda).

In [ ]:
# CÉLULA 5 — estado estendido (100% offline)
from langgraph.graph import MessagesState

class Estendido(MessagesState):
    resumo: str

be = StateGraph(Estendido)
be.add_node('n', lambda s: {'resumo': 'r', 'messages': [AIMessage(content='oi')]})
be.add_edge(START, 'n'); be.add_edge('n', END)
out = be.compile().invoke({'messages': [], 'resumo': ''})
print(out['resumo'], '|', [m.content for m in out['messages']])

## 5. Erros clássicos

1. **Paralelizar quem depende um do outro.** `n3` precisando do que `n2`
   escreveu, mas os dois na mesma etapa: `n3` lê o retrato antigo (seção 3).
   Dependência = sequência; independência = paralelo.
2. **Achar que reducer 'escolhe o melhor'.** Reducer **mescla tudo** (soma,
   concatena) — ele nunca descarta. Descarte é `overwrite` (ou `RemoveMessage`).
3. **Esquecer o `Annotated`.** Sem ele, a chave é overwrite e o fan-out morre
   com `InvalidUpdateError`. O erro diz a solução: `Use an Annotated key`.
4. **Reducer que quebra com `None`.** Estado inicial `None` + `operator.add` =
   `TypeError` (CÉLULA 3). Inicialize com `[]`/`0` ou escreva custom tolerante.
5. **Sobrescrever por `id` sem querer.** Reusar `id` de mensagem em `add_messages`
   **substitui** em vez de anexar — `id` duplicado apaga a original.
6. **Pôr reducer onde overwrite bastava.** Canal de 'último valor vence'
   (`status`, `resumo`) não precisa de `add` — reducer errado acumula lixo.

## 6. Exercício guiado — cozinha e bar em paralelo

Situação: `coletar` anota o pedido base; depois **cozinha** e **bar** produzem
em paralelo, cada um anexando seu item na mesma lista `itens`.
Sem reducer isso morre; com `Annotated[list, add]`, a comanda sai completa.

In [ ]:
# CÉLULA 6 — exercício guiado (fan-out que funciona)
from operator import add as _add
from typing import Annotated as _A, TypedDict as _TD

class Comanda(_TD):
    itens: _A[list, _add]

bc = StateGraph(Comanda)
bc.add_node('coletar', lambda s: {'itens': ['hamburguer']})
bc.add_node('cozinha', lambda s: {'itens': ['pastel']})
bc.add_node('bar', lambda s: {'itens': ['suco']})
bc.add_edge(START, 'coletar')
bc.add_edge('coletar', 'cozinha'); bc.add_edge('coletar', 'bar')
bc.add_edge('cozinha', END); bc.add_edge('bar', END)
out = bc.compile().invoke({'itens': []})
print(out)
assert sorted(out['itens']) == ['hamburguer', 'pastel', 'suco'], out
print('OK — os 3 itens, sem ninguém apagar ninguém. (Ordem entre os')
print('paralelos pode variar: por isso o sorted no assert.)')

## 7. Desafio — contador paralelo que soma

`operator.add` também soma números. Dois nós partindo de `foo=0`, cada um
devolvendo `{'foo': 1}` na mesma etapa: quanto dá no final? (Pense antes.)

- Resposta: `0 + 1 + 1 = 2` — o reducer soma o atual com cada update.
- Agora generalize: `foo: Annotated[int, add]` com 3 nós devolvendo `2` cada,
  partindo de `10`. Confira executando (esperado: `10 + 2 + 2 + 2 = 16`).
- **Pegadinha:** e se um dos nós devolver `{'foo': 0}`? Soma `0` — neutro.
  Reducer soma, não substitui: para 'zerar' use overwrite (sem `add`).

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] Por que sequência tolera overwrite e fan-out exige reducer?
- [ ] O que o `Annotated` faz e quando o `InvalidUpdateError` aparece?
- [ ] Por que o fan-out deu `[1, 2, 3, 3]` e não `[1, 2, 3]`?
- [ ] Quando escrever um reducer custom em vez de `operator.add`?
- [ ] Quais são os 3 poderes do `add_messages`?
- [ ] No seu avaliador de redação, que canais acumulariam e quais sobrescreveriam?

**Próxima aula (09 — Schemas Múltiplos):** a última do bloco — `input/output`
schemas e estado privado: o grafo recebe `pergunta`, esconde `notas` e devolve
só `resposta`.